In [ ]:
import geopandas as gpd


#Load data
quarters = gpd.read_file(r"C:\Users\luca\Downloads\Lux_public_transport_project\Lux_quaters.geojson").to_crs(epsg=2169)
commercial = gpd.read_file(r"C:\Users\luca\Downloads\Lux_public_transport_project\Commerical_land.gpkg").to_crs(epsg=2169)
stops_gpkg = gpd.read_file(r'C:\Users\luca\Downloads\Lux_public_transport_project\stops_freq.gpkg').to_crs(epsg=2169)

#intersect commecial area with quarter area and compute the result per quarter
commercial_intersect = gpd.overlay(quarters,commercial,how='intersection')
commercial_intersect['comm_area'] = commercial_intersect.area


#Create a Df summing the area per quarter
comm_area_quarter = commercial_intersect.groupby('FK_QUART_NAME')['comm_area'].sum().reset_index()

#Find the area of each quarter
quarters["total_area"] = quarters.geometry.area

#Finally, calcualte the coverage ratio. Coverage ratio = comm_area/comm_area_quarter
quarters = quarters.merge(comm_area_quarter, on='FK_QUART_NAME', how="left").fillna(0)
quarters["coverage"] = quarters["comm_area"] / quarters["total_area"]


#Select the top five coverage ratios and save them to a geodataframe
top5 = quarters.sort_values('coverage',ascending=False).head(5)

top_comm = gpd.overlay(commercial, top5, how="intersection")

#FInd candidate stops for each quarter. WE take two candidate stops within 100 meters of each polygon
stops_near = gpd.sjoin_nearest(top_comm, stops_gpkg, how="left", distance_col="dist")
stops_near = stops_near[stops_near["dist"] < 50]  
stops_comm = stops_near[["FK_QUART_NAME", "stop_id", "dist"]].drop_duplicates()

#Save top disticts and candidate stops and the top com gdf for results.
top_districts = top5[["FK_QUART_NAME", "coverage", "geometry"]]

top_districts.to_file("business_districts.gpkg", driver="GPKG")
stops_comm.to_csv("district_candidate_stops.csv", index=False)
top_comm.to_file("top_district_commercial.gpkg", driver="GPKG")




count    5.000000
mean     0.315050
std      0.165454
min      0.177922
25%      0.203042
50%      0.230827
75%      0.393716
max      0.569742
Name: coverage, dtype: float64